In [84]:
import pandas as pd
import numpy as np

import sys
sys.path.append("..")
from Src.data_modules import *

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, f1_score
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras import backend as K

# Data Loading

## training data

In [129]:
data_model = pd.read_csv("data_model.csv", sep=";", index_col=0)

y = data_model["target"]
X = data_model.drop(columns=["target"])

In [130]:
# Normaliser les données
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [131]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=False)

## testing data

In [110]:
test_data_model = pd.read_csv("test_data_model.csv", sep=";", index_col=0)

# X_test = data_model.drop(columns=["target"])

In [111]:
# Normaliser les données
scaler = StandardScaler()
test_data_model = scaler.fit_transform(test_data_model)

In [107]:
X, y = shuffle(X, y, random_state=42)

# ANN

In [132]:
def f1_metric(y_true, y_pred):
    # Convertir les deux en float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Arrondir les prédictions et les véritables valeurs
    y_true = tf.round(y_true)
    y_pred = tf.round(y_pred)
    
    # Calcul des éléments de la confusion
    tp = tf.reduce_sum(tf.cast(y_true * y_pred, tf.float32))
    fp = tf.reduce_sum(tf.cast((1 - y_true) * y_pred, tf.float32))
    fn = tf.reduce_sum(tf.cast(y_true * (1 - y_pred), tf.float32))
    
    # Calcul du F1 score
    precision = tp / (tp + fp + tf.keras.backend.epsilon())  # Ajout d'un petit epsilon pour éviter la division par zéro
    recall = tp / (tp + fn + tf.keras.backend.epsilon())
    f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon())
    
    return f1
# Construire le modèle de réseau de neurones
model = Sequential()

# Ajouter une couche d'entrée (en supposant que X_train a 10 features)
model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))  # Utilisez X_train ici, pas X

# Ajouter une couche cachée
model.add(Dense(units=32, activation='relu'))

# Ajouter une couche de sortie (pour une classification binaire ou multiclasse)
model.add(Dense(units=1, activation='sigmoid'))  # Utilisez 'softmax' pour multiclasse

# Compiler le modèle
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy', f1_metric])  # Utilisez 'categorical_crossentropy' pour multiclasse

# Entraîner le modèle
model.fit(X_train, y_train, epochs=50, batch_size=32)

# Évaluer le modèle sur l'ensemble de test
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()  # Pour une classification binaire
# Si vous avez une classification multiclasse, utilisez np.argmax(model.predict(X_test), axis=1)

# Calculer l'exactitude
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision sur l'ensemble de test : {accuracy:.4f}")

# Afficher un rapport de classification
print(classification_report(y_test, y_pred))

# Calcul du coefficient de Kappa
kappa_score = cohen_kappa_score(y_test, y_pred)

print(f"Score Kappa : {kappa_score}")

Epoch 1/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 630us/step - accuracy: 0.9380 - f1_metric: 0.9570 - loss: 0.1576
Epoch 2/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 625us/step - accuracy: 0.9515 - f1_metric: 0.9660 - loss: 0.1214
Epoch 3/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 614us/step - accuracy: 0.9524 - f1_metric: 0.9665 - loss: 0.1190
Epoch 4/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 611us/step - accuracy: 0.9546 - f1_metric: 0.9682 - loss: 0.1150
Epoch 5/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 619us/step - accuracy: 0.9536 - f1_metric: 0.9672 - loss: 0.1162
Epoch 6/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 610us/step - accuracy: 0.9552 - f1_metric: 0.9685 - loss: 0.1131
Epoch 7/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 612us/step - accuracy: 0.9555 - f1_metric: 0.9686 - loss: 0.1116
Epoch 8/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 609us/step - accuracy: 0.9550 - f1_metric: 0.9682 - loss: 0.1127
Epoch 9/50
5726/5726 ━━━━━━━━━━━━━━━━━━━━ 4s 617us/step - accuracy: 0.9553 - f1_metric: 0.9684 - loss: 0.1115
Epoch 10/5

In [18]:
# Fonction pour calculer le score Kappa après l'entraînement
def kappa_metric(y_true, y_pred):
    y_true = K.round(K.cast(y_true, 'float32'))  # On arrondit les prédictions pour avoir des classes binaires
    y_pred = K.round(K.cast(y_pred, 'float32'))  # Pour les classes binaires
    
    # On utilise Keras pour calculer la précision, mais on laisse le calcul Kappa pour après l'entraînement
    return K.mean(K.equal(y_true, y_pred), axis=-1)  # Retourne la précision pour suivre l'évolution

# Construire le modèle
model = Sequential()

# Ajouter des couches comme dans l'exemple précédent
model.add(Dense(units=64, activation='relu', input_dim=X.shape[1]))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))

# Compiler le modèle avec une fonction de perte classique et la précision comme métrique
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Entraîner le modèle
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Prédictions sur l'ensemble de test
y_pred = (model.predict(X_test) > 0.5)  # Pour une classification binaire

# Calcul du score Kappa sur les prédictions
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Score Kappa : {kappa}")

Epoch 1/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 752us/step - accuracy: 0.5336 - loss: 182.2188 - val_accuracy: 0.6552 - val_loss: 81.3801
Epoch 2/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 741us/step - accuracy: 0.6454 - loss: 38.2577 - val_accuracy: 0.6337 - val_loss: 6.4107
Epoch 3/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 759us/step - accuracy: 0.7355 - loss: 10.2224 - val_accuracy: 0.8311 - val_loss: 20.9250
Epoch 4/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 757us/step - accuracy: 0.7958 - loss: 6.3095 - val_accuracy: 0.7492 - val_loss: 1.0186
Epoch 5/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 767us/step - accuracy: 0.8007 - loss: 6.0632 - val_accuracy: 0.7947 - val_loss: 0.8770
Epoch 6/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 765us/step - accuracy: 0.8386 - loss: 4.9068 - val_accuracy: 0.8415 - val_loss: 21.1576
Epoch 7/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 796us/step - accuracy: 0.8595 - loss: 6.0571 - val_accuracy: 0.8794 - val_loss: 0.5291
Epoch 8/10
6544/6544 ━━━━━━━━━━━━━━━━━━━━ 5s 745us/step - accuracy: 0

# Submission

In [109]:
creat_submission_file(test_data_model, model, "submission_ann")